# Modul 7 · Lineare Regression
**Data Science 1** · linkit courses

Wie viel Ausbeute kostet eine Stunde Stillstand? Ist die Nachtschicht bei gleicher Auslastung schlechter, und wirkt mehr Personal wirklich? In Modul 4 hast du diese Fragen mit Schichtung angedeutet. Hier baust du Regressionsmodelle mit mehreren Einflüssen, liest ihre Koeffizienten mit Standardfehlern, prüfst die Annahmen an den Residuen und testest Vorhersagen an einem Jahr, das das Modell nicht kennt.

**Lernziele**
1. Eine Regressionsgerade mit kleinsten Quadraten bestimmen und deuten
2. Koeffizienten einer multiplen Regression als „bei sonst gleichen“ Effekte lesen
3. Kategoriale Merkmale und Wechselwirkungen in ein Modell aufnehmen
4. Standardfehler, Intervalle und Tests der Koeffizienten mit statsmodels bestimmen
5. Annahmen mit Residuen prüfen und Probleme erkennen
6. Vorhersagen mit Intervallen machen und außerhalb der Stichprobe prüfen
7. Erklären, warum ein Regressionskoeffizient keine Ursache beweist

**So arbeitest du**
- Das Notebook läuft auf der Kursseite direkt im Browser, ohne Installation. Alternativ öffnest du es in Google Colab und speicherst dort zuerst eine eigene Kopie (*Datei → Kopie in Drive speichern*).
- Führe die Zellen von oben nach unten mit **Shift + Enter** aus. Der erste Import von statsmodels lädt im Browser SciPy, patsy und statsmodels und dauert einmalig etwa 10 Sekunden.
- Stellen mit `# TODO` füllst du selbst aus, meist mit ein bis zehn Zeilen. Darunter prüft ein Selbsttest dein Ergebnis. Ein ❌ ist nur ein Hinweis, dass etwas noch nicht stimmt, das Notebook läuft trotzdem weiter.
- Zellen, die ein Ergebnis aus einem TODO brauchen, melden „⏳ Erst TODO n lösen.“, solange es fehlt.
- Mindestziel sind TODO 1 bis 6, TODO 8 sowie TODO 10 bis 13. TODO 7, TODO 9 und die Bonusaufgaben vertiefen.
- TODO-Zellen mit Zufall legen ihren eigenen Generator mit festem Seed an, `rng = np.random.default_rng(...)`. Die Selbsttests lassen so viel Spielraum, dass jeder richtige Weg besteht.
- Gib Ergebnisse von statsmodels mit `float(...)`, `round(...)` oder der Hilfsfunktion `zeige` aus, dann steht im Browser und in Colab dasselbe da. Jede Grafikzelle endet mit `plt.show()`.
- Jeder Abschnitt gehört zu einer Lerneinheit und zum Abschnitt mit derselben Nummer im Skript. Hilfen zu jedem TODO und typische Fehler findest du auf der Kursseite.

In [ ]:
# Hilfsfunktion für die Selbsttests: einfach ausführen, nicht ändern.
import math

def check(name, test):
    """Führt einen Test aus und zeigt ✅ oder ❌. Fehler im Test gelten als „noch nicht gelöst“."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

print("Bereit.")

In [ ]:
# Daten laden: Auf der Kursseite liegen die Dateien schon bereit, in Colab holt diese Zelle sie von GitHub.
import os, urllib.request
DATEN = ["produktion.csv", "stoerungen.csv", "maschinen.csv", "linien.csv"]
QUELLE = "https://raw.githubusercontent.com/juliankunz18/linkit-courses/main/data-science-1/daten/"
os.makedirs("daten", exist_ok=True)
for datei in DATEN:
    if not os.path.exists("daten/" + datei):
        urllib.request.urlretrieve(QUELLE + datei, "daten/" + datei)
print("Daten bereit:", ", ".join(sorted(os.listdir("daten"))))

## 1 · Eine Gerade durch die Punktwolke
*Lerneinheit 1 · Skript Abschnitt 1*

Das einfache lineare Modell $y_i = \beta_0 + \beta_1 x_i + \varepsilon_i$ wird mit der **Methode der kleinsten Quadrate** geschätzt: Die Summe der quadrierten Residuen $e_i = y_i - \hat y_i$ soll so klein wie möglich sein. Das ergibt

$$\hat\beta_1 = \frac{\sum_i (x_i - \bar x)(y_i - \bar y)}{\sum_i (x_i - \bar x)^2} = r\,\frac{s_y}{s_x}, \qquad \hat\beta_0 = \bar y - \hat\beta_1 \bar x, \qquad R^2 = 1 - \frac{\text{SSE}}{\text{SST}}.$$

Die nächste Zelle baut die Analysetabelle `schichten` wie im Skript: eine Zeile je Linie, Tag und Schicht mit dem Stillstand aus den Störungen, der Ausschussquote `quote` in Prozent und der Ausbeute `ausbeute`. Dazu kommen die Hilfsspalten, die das Skript später anlegt: `nacht` (1 für Nachtschichten), die zentrierten Größen `temp_z` und `ausl_z` und die Kalenderwoche `kw`. Neu ist statsmodels mit `smf.ols("ziel ~ einfluss", data=...)`. Die Hilfsfunktion `zeige` druckt Zahlen mit 6 gültigen Stellen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

SCHICHT_TYP = pd.CategoricalDtype(["Früh", "Spät", "Nacht"], ordered=True)
produktion = pd.read_csv("daten/produktion.csv", dtype={"schicht": SCHICHT_TYP})
stoerungen = pd.read_csv("daten/stoerungen.csv", dtype={"schicht": SCHICHT_TYP})
maschinen = pd.read_csv("daten/maschinen.csv")
linien = pd.read_csv("daten/linien.csv")
st_linie = stoerungen.merge(maschinen[["maschine", "linie"]], on="maschine", validate="many_to_one")
still = st_linie.groupby(["datum", "linie", "schicht"], observed=True)["stillstand_min"].sum()
schichten = produktion.merge(still.reset_index(), on=["datum", "linie", "schicht"], how="left", validate="one_to_one")
schichten["stillstand_min"] = schichten["stillstand_min"].fillna(0)
schichten["quote"] = schichten["ausschuss_stueck"] / schichten["ist_stueck"] * 100
schichten["ausbeute"] = schichten["ist_stueck"] / schichten["soll_stueck"]
schichten["nacht"] = (schichten["schicht"] == "Nacht").astype(float)
schichten["temp_z"] = schichten["hallentemp_c"] - 21
schichten["ausl_z"] = schichten["auslastung"] - 0.85
schichten["kw"] = pd.to_datetime(schichten["datum"]).dt.strftime("%V")
print(schichten.shape, int((schichten["stillstand_min"] == 0).sum()), int((schichten["quote"] == 0).sum()))


def zeige(*werte):
    """Druckt Zahlen mit 6 gültigen Stellen, damit im Browser und in Colab dieselben Ziffern stehen. None bleibt None."""
    def text(w):
        if w is None or isinstance(w, (str, bool)):
            return str(w)
        if np.ndim(w) == 0:
            return f"{float(w):.6g}"
        return "[" + " ".join(f"{float(v):.6g}" for v in np.ravel(w)) + "]"
    print(*[text(w) for w in werte])

Linie 1 wie im Skript: Ausbeute gegen Stillstand in Minuten, erst mit den Formeln, dann mit `np.polyfit` und statsmodels.

In [ ]:
l1 = schichten[schichten["linie"] == 1]
x, y = l1["stillstand_min"], l1["ausbeute"]
b1 = ((x - x.mean()) * (y - y.mean())).sum() / ((x - x.mean()) ** 2).sum()
b0 = y.mean() - b1 * x.mean()
print(len(l1), round(b0, 4), round(b1, 6), np.polyfit(x, y, 1).round(6).tolist())
m_l1 = smf.ols("ausbeute ~ stillstand_min", data=l1).fit()
print(m_l1.params.round(6))

### TODO 1 · Eine Gerade von Hand für Linie 3
Linie 3 schweißt Baugruppen. Kostet Stillstand dort genauso viel wie in der Wellenfertigung? `t3` enthält die Schichten von Linie 3.
1. `b1_l3` und `b0_l3`: Steigung und Achsenabschnitt der Geraden Ausbeute gegen `stillstand_min` mit den Formeln oben.
2. `poly_l3`: das Ergebnis von `np.polyfit` für dieselbe Gerade. `m_l3`: das angepasste Modell `smf.ols("ausbeute ~ stillstand_min", data=t3).fit()`.
3. `kosten_l3`: was eine Stunde Stillstand in Linie 3 kostet, in Prozentpunkten der geplanten Stückzahl, als positive Zahl.

In [ ]:
t3 = schichten[schichten["linie"] == 3]

# TODO 1a
b1_l3 = None
b0_l3 = None

# TODO 1b und 1c
poly_l3 = None
m_l3 = None
kosten_l3 = None

zeige(b1_l3, b0_l3, poly_l3, kosten_l3)

In [ ]:
check("TODO 1a Steigung und Achsenabschnitt", lambda: len(t3) == 1447 and math.isclose(b1_l3, -0.00100350818, abs_tol=1e-10)
      and math.isclose(b0_l3, 0.949991008, abs_tol=1e-8))
check("TODO 1b wie polyfit und statsmodels", lambda: np.allclose(poly_l3, [b1_l3, b0_l3], rtol=1e-9, atol=1e-12)
      and np.allclose(m_l3.params.to_numpy(), [b0_l3, b1_l3], rtol=1e-9, atol=1e-12))
check("TODO 1c Kosten je Stunde", lambda: math.isclose(kosten_l3, 6.0210491, abs_tol=1e-5))

In Linie 3 kostet eine Stunde Stillstand 6,02 Prozentpunkte der geplanten Stückzahl, in Linie 1 waren es 4,75. Bei 0 Minuten sagen beide Geraden 95,0 % voraus. Die 231 Schichten von Linie 3 ohne gemeldete Störung kamen im Mittel auf 95,7 %, die 73 von Linie 1 auf 96,1 %: Am Rand liegen beide Geraden etwas zu tief. Solche Abweichungen findest du mit den Residuen in Abschnitt 5. Ob der Unterschied zwischen den Steigungen mehr ist als Zufall, prüfst du in TODO 9.

**Bestimmtheitsmaß.** $\text{SST} = \sum_i (y_i - \bar y)^2$ ist die Gesamtstreuung, $\text{SSE} = \sum_i e_i^2$ die Streuung der Residuen. Für Linie 1 wie im Skript:

In [ ]:
r = x.corr(y)
sst = ((y - y.mean()) ** 2).sum()
sse = (m_l1.resid ** 2).sum()
print(round(r, 4), round(r * y.std() / x.std(), 6), round(1 - sse / sst, 4), round(r ** 2, 4), round(m_l1.rsquared, 4))

### TODO 2 · Bestimmtheitsmaß von Hand für Linie 3
1. `sst_l3` und `sse_l3`: Gesamtstreuung und Quadratsumme der Residuen für die Gerade aus TODO 1. Rechne die Residuen mit der Formel `t3["ausbeute"] - (b0 + b1 * t3["stillstand_min"])` aus Steigung und Achsenabschnitt, die du hier mit `np.polyfit` neu bestimmst.
2. `r2_l3`: das Bestimmtheitsmaß, und `r_l3`: der Korrelationskoeffizient nach Pearson von Stillstand und Ausbeute in Linie 3.

In [ ]:
# TODO 2a
sst_l3 = None
sse_l3 = None

# TODO 2b
r2_l3 = None
r_l3 = None

zeige(sst_l3, sse_l3, r2_l3, r_l3)

In [ ]:
check("TODO 2a Quadratsummen", lambda: math.isclose(sst_l3, 11.1077426, abs_tol=1e-6) and math.isclose(sse_l3, 3.02777975, abs_tol=1e-6))
check("TODO 2b Bestimmtheitsmaß", lambda: math.isclose(r2_l3, 0.72741718, abs_tol=1e-7) and math.isclose(r_l3, -0.85288756, abs_tol=1e-7)
      and math.isclose(r_l3 ** 2, r2_l3, rel_tol=1e-9))

Der Stillstand erklärt in Linie 3 knapp 73 % der Streuung der Ausbeute, $r = -0{,}853$. Wer $\text{SSE}/\text{SST}$ ausrechnet, bekommt 0,27, den unerklärten Anteil.

## 2 · Mehrere Einflüsse gleichzeitig
*Lerneinheit 2 · Skript Abschnitt 2*

In Matrixform ist $y = X\beta + \varepsilon$, die Designmatrix $X$ hat eine Spalte aus Einsen für den Achsenabschnitt. Die Kleinste-Quadrate-Lösung erfüllt die **Normalgleichungen** $X^\top X\,\hat\beta = X^\top y$. `np.linalg.lstsq(X, y, rcond=None)[0]` löst sie direkt aus $X$, ohne die Inverse. Jeder Koeffizient gilt **bei sonst gleichen** übrigen Einflussgrößen. Lässt du eine Größe $x_2$ weg, verschiebt sich der Koeffizient von $x_1$ um die **Verzerrung durch ausgelassene Variablen**:

$$\hat\beta_1^{\text{kurz}} = \hat\beta_1^{\text{lang}} + \hat\beta_2^{\text{lang}}\,\hat\delta, \qquad \hat\delta = \text{Steigung von } x_2 \text{ auf } x_1.$$

Nacht und Auslastung und das Personal mit und ohne Linie wie im Skript:

In [ ]:
X = np.column_stack([np.ones(len(schichten)), schichten["nacht"], schichten["auslastung"]])
beta, *_ = np.linalg.lstsq(X, schichten["quote"], rcond=None)
m_n2 = smf.ols("quote ~ nacht + auslastung", data=schichten).fit()
print(beta.round(4), m_n2.params.round(4).to_numpy())

m_n1 = smf.ols("quote ~ nacht", data=schichten).fit()
m_n3 = smf.ols("quote ~ nacht + auslastung + C(linie)", data=schichten).fit()
print(round(m_n1.params["nacht"], 3), round(m_n2.params["nacht"], 3), round(m_n3.params["nacht"], 3), round(m_n3.params["auslastung"], 3))

m_p1 = smf.ols("quote ~ personal", data=schichten).fit()
m_p2 = smf.ols("quote ~ personal + C(linie)", data=schichten).fit()
print(round(m_p1.params["personal"], 4), round(m_p2.params["personal"], 4), m_p2.conf_int().loc["personal"].round(4).tolist())

delta = smf.ols("auslastung ~ nacht", data=schichten).fit().params["nacht"]
print(round(delta, 4), round(m_n2.params["auslastung"], 4))
print(round(m_n2.params["nacht"] + m_n2.params["auslastung"] * delta, 4), round(m_n1.params["nacht"], 4))

### TODO 3 · Normalgleichungen für Linie 1
In Linie 1 hängen Auslastung und Personal zusammen. Wie hängt die Quote von beiden zugleich ab?
1. `X_l1`: die Designmatrix für die Schichten `l1` mit einer Spalte aus Einsen, der Auslastung und dem Personal, in dieser Reihenfolge. `beta_lstsq`: die Lösung mit `np.linalg.lstsq`. `beta_normal`: die Lösung der Normalgleichungen mit `np.linalg.solve`.
2. `m_ap`: das Modell `quote ~ auslastung + personal` für `l1` mit statsmodels.
3. `r2_ap` und `r2_adj_ap`: $R^2$ und das adjustierte $R^2 = 1 - (1 - R^2)(n - 1)/(n - p - 1)$ von Hand aus den Residuen, mit $p = 2$ Einflussgrößen.

In [ ]:
# TODO 3a
X_l1 = None
beta_lstsq = None
beta_normal = None

# TODO 3b
m_ap = None

# TODO 3c
r2_ap = None
r2_adj_ap = None

zeige(beta_lstsq, beta_normal, r2_ap, r2_adj_ap)

In [ ]:
check("TODO 3a Designmatrix und lstsq", lambda: X_l1.shape == (1447, 3) and np.allclose(X_l1[:, 0], 1)
      and np.allclose(beta_lstsq, [1.4332135, 1.8142188, -0.0600845], atol=1e-6))
check("TODO 3a Normalgleichungen", lambda: np.allclose(beta_normal, beta_lstsq, rtol=1e-8, atol=1e-10))
check("TODO 3b wie statsmodels", lambda: np.allclose(m_ap.params.to_numpy(), beta_lstsq, rtol=1e-8, atol=1e-10))
check("TODO 3c Bestimmtheitsmaß und adjustiert", lambda: math.isclose(r2_ap, 0.00984905, abs_tol=1e-7)
      and math.isclose(r2_adj_ap, 0.00847765, abs_tol=1e-7) and math.isclose(r2_adj_ap, m_ap.rsquared_adj, abs_tol=1e-10))

Bei gleichem Personal geht 0,1 mehr Auslastung mit 0,18 Prozentpunkten mehr Ausschuss einher, bei gleicher Auslastung jede Person mit 0,06 weniger. Beide Größen erklären in Linie 1 aber nur 1 % der Streuung, das adjustierte $R^2$ liegt mit 0,0085 knapp darunter. Ohne die Einsen in `X_l1` erzwingt `lstsq` eine Ebene durch den Nullpunkt, und die Koeffizienten werden ganz andere.

### TODO 4 · Verzerrung durch eine ausgelassene Variable
Was passiert mit dem Koeffizienten der Auslastung, wenn das Personal fehlt? Rechne für Linie 1:
1. `m_lang`: das lange Modell `quote ~ auslastung + personal`, `b_kurz`: die Steigung der Auslastung im kurzen Modell `quote ~ auslastung`.
2. `delta_ap`: die Steigung aus der Regression von `personal` auf `auslastung`. `verzerrung`: der Koeffizient des Personals im langen Modell mal `delta_ap`.

In [ ]:
# TODO 4a
m_lang = None
b_kurz = None

# TODO 4b
delta_ap = None
verzerrung = None

zeige(b_kurz, delta_ap, verzerrung)

In [ ]:
check("TODO 4a kurzes Modell", lambda: math.isclose(b_kurz, 1.23646856, abs_tol=1e-7)
      and math.isclose(m_lang.params["auslastung"], 1.81421879, abs_tol=1e-7))
check("TODO 4b Formel exakt", lambda: math.isclose(delta_ap, 9.61563417, abs_tol=1e-6) and math.isclose(verzerrung, -0.57775023, abs_tol=1e-6)
      and math.isclose(m_lang.params["auslastung"] + verzerrung, b_kurz, abs_tol=1e-9))

In [ ]:
if verzerrung is None:
    print("⏳ Erst TODO 4 lösen.")
else:
    print(f"lang {m_lang.params['auslastung']:.3f} + Verzerrung {verzerrung:.3f} = kurz {b_kurz:.3f}")
    print(f"Korrelation von Personal und Auslastung in Linie 1: {l1['personal'].corr(l1['auslastung']):.3f}")

Je 1,0 mehr Auslastung hat eine Schicht in Linie 1 im Mittel 9,6 Beschäftigte mehr, und bei gleicher Auslastung geht mehr Personal mit weniger Ausschuss einher. Ohne Personal im Modell zieht das den Koeffizienten der Auslastung von 1,81 auf 1,24, um genau $-0{,}578$. Welcher der beiden Koeffizienten die Frage beantwortet, entscheidet das Modul mit Abschnitt 7: Das Personal richtet sich nach dem Plan.

## 3 · Kategorien und Wechselwirkungen
*Lerneinheit 3 · Skript Abschnitt 3*

Eine Kategorie geht als **Dummies** ein, je eine 0-1-Spalte außer für die **Referenzkategorie**. Ein Dummy-Koeffizient ist der Abstand zur Referenz. `C(schicht)` nimmt Früh als Referenz, `C(schicht, Treatment(reference="Nacht"))` die Nacht. Mit allen Kategorien und der Spalte aus Einsen sind die Spalten perfekt kollinear, die **Dummy-Falle**. Eine **Wechselwirkung** `a * b` lässt die Steigung von `a` je Kategorie von `b` verschieden sein: Die Steigung einer Kategorie ist der Haupteffekt plus ihr Wechselwirkungskoeffizient. Das Hauptmodell des Skripts mit zentrierter Temperatur und Auslastung:

In [ ]:
m_lin = smf.ols("quote ~ C(linie)", data=schichten).fit()
m_lin4 = smf.ols("quote ~ C(linie, Treatment(reference=4))", data=schichten).fit()
print(m_lin.params.round(4).to_numpy(), m_lin4.params.round(4).to_numpy())

m_t = smf.ols("quote ~ hallentemp_c * C(linie) + auslastung + C(schicht)", data=schichten).fit()
b = m_t.params
steigung = pd.Series({1: b["hallentemp_c"]} | {l: b["hallentemp_c"] + b[f"hallentemp_c:C(linie)[T.{l}]"] for l in [2, 3, 4]})
print(steigung.round(4).to_dict())

m_z = smf.ols("quote ~ temp_z * C(linie) + ausl_z + C(schicht)", data=schichten).fit()
m_ohne_temp = smf.ols("quote ~ C(linie) + ausl_z + C(schicht)", data=schichten).fit()
print(round(m_ohne_temp.params["C(schicht)[T.Nacht]"], 3), round(m_z.params["C(schicht)[T.Nacht]"], 3))

### TODO 5 · Die Nacht als Referenz
1. `X_schicht`: eine Designmatrix für die Schichten mit einer Spalte aus Einsen und Dummies für Früh und Spät, die Nacht ist Referenz. `pd.get_dummies(schichten["schicht"], dtype=float)` liefert alle drei Dummies. `beta_schicht`: die Lösung von `lstsq` für die Quote.
2. `m_nacht_ref`: das Modell `quote ~ C(schicht, Treatment(reference="Nacht")) + C(linie) + ausl_z` mit statsmodels. `frueh_gegen_nacht`: sein Koeffizient für Früh.

In [ ]:
# TODO 5a
X_schicht = None
beta_schicht = None

# TODO 5b
m_nacht_ref = None
frueh_gegen_nacht = None

zeige(beta_schicht, frueh_gegen_nacht)

In [ ]:
mittel_schicht = schichten.groupby("schicht", observed=True)["quote"].mean()
check("TODO 5a Abstände zur Nacht", lambda: X_schicht.shape == (5788, 3) and np.linalg.matrix_rank(X_schicht) == 3
      and np.allclose(beta_schicht, [mittel_schicht["Nacht"], mittel_schicht["Früh"] - mittel_schicht["Nacht"], mittel_schicht["Spät"] - mittel_schicht["Nacht"]], atol=1e-9))
check("TODO 5b Früh gegen Nacht bei gleicher Linie und Auslastung", lambda: math.isclose(frueh_gegen_nacht, -0.3203521, abs_tol=1e-6)
      and math.isclose(m_nacht_ref.params["ausl_z"], 2.7144303, abs_tol=1e-6))

Ohne weitere Größen ist der Achsenabschnitt die mittlere Quote der Nachtschichten, 2,06 %, die Frühschicht liegt 0,11 Prozentpunkte darunter. Bei gleicher Linie und Auslastung sind es 0,32: Die Nacht ist geringer ausgelastet, das verdeckt einen Teil ihres Nachteils. Mit allen drei Dummies und den Einsen hat `X_schicht` nur Rang 3 von 4 Spalten, und `lstsq` liefert ohne Warnung Koeffizienten, die keine Abstände sind.

### TODO 6 · Kostet Stillstand in jeder Linie gleich viel?
1. `m_aus`: das Modell `ausbeute ~ stillstand_min * C(linie)` für alle Schichten.
2. `steigung_aus`: eine Series mit den Linien 1 bis 4 als Index und der Steigung des Stillstands je Minute in jeder Linie, zusammengesetzt aus den Koeffizienten. `kosten_aus`: dieselben Steigungen als Kosten einer Stunde Stillstand in Prozentpunkten, positiv.

In [ ]:
# TODO 6a
m_aus = None

# TODO 6b
steigung_aus = None
kosten_aus = None

print(None if kosten_aus is None else kosten_aus.round(3).to_dict())

In [ ]:
check("TODO 6a Wechselwirkung mit Kategorie", lambda: len(m_aus.params) == 8 and "stillstand_min:C(linie)[T.3]" in m_aus.params.index)
check("TODO 6b Steigungen je Linie", lambda: list(steigung_aus.index) == [1, 2, 3, 4]
      and np.allclose(steigung_aus.to_numpy(), [-0.000791733, -0.000796034, -0.001003508, -0.000668648], atol=1e-9))
check("TODO 6b Kosten je Stunde", lambda: np.allclose(kosten_aus.to_numpy(), [4.7504, 4.7762, 6.0210, 4.0119], atol=1e-3))

Eine Stunde Stillstand kostet die Linien 1 und 2 je etwa 4,8 Prozentpunkte, Linie 3 6,0 und Linie 4 4,0. Die Steigungen je Linie sind genau die aus vier getrennten Regressionen, denn das Modell enthält nur Stillstand und Linie. Ohne Wechselwirkung, mit `stillstand_min + C(linie)`, bekäme jede Linie dieselbe Steigung. Mit `linie` ohne `C()` wäre die Liniennummer eine Zahl, und die Steigung änderte sich gleichmäßig von Linie zu Linie.

### TODO 7 · Logarithmus und Prozent
Das Log-Modell des Skripts lässt die Schichten ohne Ausschuss weg, denn $\ln 0$ ist nicht definiert. Hier schätzt du es mit den rohen Größen statt der zentrierten.
1. `m_log_roh`: das Modell `np.log(quote) ~ hallentemp_c * C(linie) + auslastung + C(schicht)` für die Schichten mit `quote > 0` (`mit_ausschuss`). `n_log`: die Zahl der Schichten, die es nutzt.
2. `nacht_prozent`: um wie viel Prozent die Quote nachts höher liegt als früh, $100\,(e^{b} - 1)$. `ausl_prozent`: um wie viel Prozent sie mit 0,1 mehr Auslastung steigt.
3. `quote_l4_alle` und `quote_l4_mit`: die mittlere Quote von Linie 4 über alle Schichten und nur über die Schichten mit Ausschuss.

In [ ]:
mit_ausschuss = schichten[schichten["quote"] > 0]

# TODO 7a
m_log_roh = None
n_log = None

# TODO 7b
nacht_prozent = None
ausl_prozent = None

# TODO 7c
quote_l4_alle = None
quote_l4_mit = None

zeige(n_log, nacht_prozent, ausl_prozent, quote_l4_alle, quote_l4_mit)

In [ ]:
check("TODO 7a Schichten mit Ausschuss", lambda: n_log == 5320 and math.isclose(m_log_roh.params["hallentemp_c"], 0.0822761, abs_tol=1e-6)
      and math.isclose(m_log_roh.params["Intercept"], -2.4721411, abs_tol=1e-6) and math.isclose(m_log_roh.params["C(linie)[T.3]"], 1.0210065, abs_tol=1e-6))
check("TODO 7b Effekte in Prozent", lambda: math.isclose(nacht_prozent, 27.4518, abs_tol=1e-3) and math.isclose(ausl_prozent, 17.5363, abs_tol=1e-3))
check("TODO 7c Auswahl in Linie 4", lambda: math.isclose(quote_l4_alle, 0.976481, abs_tol=1e-5) and math.isclose(quote_l4_mit, 1.329227, abs_tol=1e-5))

Steigungen und Schichtkoeffizienten gleichen denen aus dem Skript, die Temperatur bringt in Linie 1 wieder 0,082 je Grad. Achsenabschnitt und Liniendummies ändern sich, denn mit den rohen Größen beschreiben sie eine Schicht bei 0 °C und Auslastung 0. Wegen der Wechselwirkung liegt Linie 3 dort auf der Log-Skala 1,02 über Linie 1, bei 21 °C dagegen 0,25 darunter. Nachts liegt die Quote im Log-Modell um 27 % höher als früh, 0,1 mehr Auslastung bringt 18 % mehr. Die Prozente gelten aber nur für Schichten mit Ausschuss. In Linie 4 fehlt jede vierte Schicht, und die übrigen haben im Mittel 1,33 % statt 0,98 % Ausschuss. Wer `np.log(quote)` auf alle Schichten anwendet, bekommt für die Nullen $-\infty$, und alle Koeffizienten werden ohne Fehlermeldung `nan`.

## 4 · Wie sicher sind die Koeffizienten?
*Lerneinheit 4 · Skript Abschnitt 4*

Die Standardfehler sind die Wurzeln aus der Diagonale von $\hat\sigma^2 (X^\top X)^{-1}$ mit $\hat\sigma^2 = \text{SSE}/(n - p - 1)$. Robuste Standardfehler nach White (1980) ersetzen $\hat\sigma^2$ durch die einzelnen $e_i^2$:

$$(X^\top X)^{-1}\, X^\top \mathrm{diag}(w_i)\, X\, (X^\top X)^{-1}, \qquad w_i = \frac{e_i^2}{(1 - h_i)^2} \text{ für HC3}.$$

$h_i$ ist der Hebel, die Diagonale von $X (X^\top X)^{-1} X^\top$. Der **F-Test** prüft $q$ Koeffizienten zugleich: $F = \big((\text{SSE}_0 - \text{SSE}_1)/q\big)/\big(\text{SSE}_1/(n - p - 1)\big)$. Wie im Skript:

In [ ]:
X1 = np.column_stack([np.ones(len(l1)), l1["stillstand_min"]])
b_hand = np.linalg.lstsq(X1, l1["ausbeute"], rcond=None)[0]
e_hand = l1["ausbeute"] - X1 @ b_hand
sigma2 = (e_hand ** 2).sum() / (len(l1) - 2)
se_hand = np.sqrt(sigma2 * np.diag(np.linalg.inv(X1.T @ X1)))
print(f"{se_hand[0]:.6f} {se_hand[1]:.8f}", f"{m_l1.bse.iloc[0]:.6f} {m_l1.bse.iloc[1]:.8f}")

tabelle = pd.DataFrame({"b": m_z.params, "se": m_z.bse, "unten": m_z.conf_int()[0], "oben": m_z.conf_int()[1], "p": m_z.pvalues})
print(tabelle.round(4))

from statsmodels.stats.anova import anova_lm
m_ohne_ww = smf.ols("quote ~ temp_z + C(linie) + ausl_z + C(schicht)", data=schichten).fit()
f_hand = ((m_ohne_ww.ssr - m_z.ssr) / 3) / (m_z.ssr / m_z.df_resid)
print(round(m_ohne_ww.ssr, 2), round(m_z.ssr, 2), round(f_hand, 2))
print(anova_lm(m_ohne_ww, m_z).round(4))
f_test = m_z.f_test("temp_z:C(linie)[T.2] = 0, temp_z:C(linie)[T.3] = 0, temp_z:C(linie)[T.4] = 0")
print(round(float(f_test.fvalue), 2), f"{float(f_test.pvalue):.1e}")

m_hc3 = smf.ols("quote ~ temp_z * C(linie) + ausl_z + C(schicht)", data=schichten).fit(cov_type="HC3")
print(pd.DataFrame({"klassisch": m_z.bse, "HC3": m_hc3.bse, "Verhältnis": m_hc3.bse / m_z.bse}).round(4))

### TODO 8 · Standardfehler von Hand, klassisch und robust
Das Modell `quote ~ auslastung + personal` für Linie 1 aus TODO 3. `X_se` und `y_se` stehen schon da.
1. `beta_se`: die Koeffizienten mit `lstsq`. `se_klassisch`: die drei klassischen Standardfehler mit $\hat\sigma^2 = \text{SSE}/(n - p - 1)$.
2. `hebel_se`: die Hebelwerte, die Diagonale von $X (X^\top X)^{-1} X^\top$. Tipp: `(X_se @ xtx_inv * X_se).sum(axis=1)` rechnet nur die Diagonale. `se_hc3`: die robusten Standardfehler nach HC3. `m_hc3_se`: dasselbe Modell mit `smf.ols(...).fit(cov_type="HC3")` zum Vergleich.
3. Simulation als Gegenprobe: Erzeuge mit `rng` 2.000 neue Zielgrößen $X\hat\beta + \varepsilon$ mit normalverteilten Fehlern der Standardabweichung $\hat\sigma$, als Matrix der Form (1447, 2000), schätze alle mit einem Aufruf von `lstsq` und speichere in `sd_sim` die Standardabweichung der 2.000 Koeffizienten der Auslastung.

In [ ]:
X_se = np.column_stack([np.ones(len(l1)), l1["auslastung"], l1["personal"]])
y_se = l1["quote"].to_numpy()

# TODO 8a
beta_se = None
se_klassisch = None

# TODO 8b
hebel_se = None
se_hc3 = None
m_hc3_se = None

# TODO 8c
rng = np.random.default_rng(8)
sd_sim = None

zeige(se_klassisch, se_hc3, sd_sim)

In [ ]:
check("TODO 8a klassische Standardfehler", lambda: np.allclose(beta_se, [1.4332135, 1.8142188, -0.0600845], atol=1e-6)
      and np.allclose(se_klassisch, [0.35833455, 0.48301628, 0.02558568], atol=2e-7))
check("TODO 8b Hebel", lambda: len(hebel_se) == 1447 and math.isclose(hebel_se.sum(), 3.0, abs_tol=1e-9))
check("TODO 8b HC3 wie statsmodels", lambda: np.allclose(se_hc3, [0.36692367, 0.50077247, 0.02602137], atol=2e-7)
      and np.allclose(se_hc3, m_hc3_se.bse.to_numpy(), rtol=1e-8))
check("TODO 8c Simulation", lambda: abs(sd_sim - 0.483016) <= 0.035)

Die Auslastung hat den Standardfehler 0,483, robust 0,501. Die Simulation trifft den klassischen Wert, denn sie erzeugt Fehler mit gleicher Varianz, genau wie die Formel annimmt. Die Hebelwerte summieren sich zur Zahl der Koeffizienten, 3. Mit $\text{SSE}/n$ statt $\text{SSE}/(n - 3)$ wären die Standardfehler um 0,1 % zu klein, bei kleinen Stichproben um viel mehr.

### TODO 9 · F-Test: Kostet Stillstand überall gleich viel?
1. `m_aus0` und `m_aus1`: die Modelle `ausbeute ~ stillstand_min + C(linie)` (gleiche Steigung) und `ausbeute ~ stillstand_min * C(linie)` (Steigung je Linie).
2. `f_aus`: die F-Statistik von Hand aus den beiden Quadratsummen (`.ssr`) und `.df_resid`, `p_aus`: der p-Wert mit `stats.f.sf(f_aus, q, df)` aus `from scipy import stats`. `anova_aus`: der Vergleich mit `anova_lm`.

In [ ]:
from scipy import stats

# TODO 9a
m_aus0 = None
m_aus1 = None

# TODO 9b
f_aus = None
p_aus = None
anova_aus = None

zeige(f_aus, p_aus)

In [ ]:
check("TODO 9a Modelle", lambda: len(m_aus0.params) == 5 and len(m_aus1.params) == 8 and int(m_aus1.df_resid) == 5780)
check("TODO 9b F von Hand", lambda: math.isclose(f_aus, 76.83278, abs_tol=1e-4) and 0 < p_aus < 1e-40)
check("TODO 9b wie anova_lm", lambda: math.isclose(float(anova_aus["F"].iloc[1]), f_aus, rel_tol=1e-9))

$F = 76{,}8$ mit 3 und 5.780 Freiheitsgraden, $p \approx 10^{-48}$: Die Steigungen unterscheiden sich, vor allem durch Linie 3. Bei so vielen Schichten wird auch ein kleiner Unterschied nachweisbar. Ob er zählt, sagen die Kosten aus TODO 6.

## 5 · Residuen lesen: Annahmen und Diagnose
*Lerneinheit 5 · Skript Abschnitt 5*

Residuen gegen Vorhersage zeigen Kurven und Trichter, Residuen im Zeitverlauf fehlende Größen, Hebel und **Cook-Abstand** einflussreiche Schichten. Der **Varianzinflationsfaktor** $\mathrm{VIF}_j = 1/(1 - R_j^2)$ misst Kollinearität, mit $R_j^2$ aus der Regression von $x_j$ auf die übrigen Einflussgrößen. Für das Hauptmodell wie im Skript:

In [ ]:
klasse = pd.qcut(m_z.fittedvalues, 5, labels=False)
print(m_z.resid.groupby(klasse).agg(["mean", "std"]).round(3))

monat = schichten["datum"].str[:7]
jahresgang = pd.DataFrame({"ohne": m_ohne_temp.resid.groupby(monat).mean(), "mit": m_z.resid.groupby(monat).mean()})
print(jahresgang.std().round(3).to_dict())

einfluss = m_z.get_influence()
cook, hebel = einfluss.cooks_distance[0], einfluss.hat_matrix_diag
i_max, i_hebel = int(np.argmax(cook)), int(np.argmax(hebel))
print(round(hebel.mean(), 5), round(float(hebel.max()), 4), round(float(cook.max()), 4))

### TODO 10 · Diagnose der Ausbeute
Wie gut passt das Modell `ausbeute ~ stillstand_min * C(linie)`? `m_diag` steht schon da.
1. `res_klassen`: ein DataFrame mit Mittelwert (`mean`) und Standardabweichung (`std`) der Residuen in fünf gleich großen Klassen der Vorhersage, wie oben. `res_monat`: eine Series mit dem mittleren Residuum je Monat. Zeichne am Ende der Zelle in `fig_d, axes_d` links die Residuen gegen die Vorhersage, rechts die Monatsmittel.
2. `vif_ausl`: der VIF der Auslastung im Modell `quote ~ auslastung + personal` für Linie 1 aus TODO 3, von Hand über die Hilfsregression.
3. Für das Modell `m_l3_diag` der Linie 3: `i_cook`, die Position (0 bis 1446) der Schicht mit dem größten Cook-Abstand in `t3`, `cook_max`, ihr Cook-Abstand, und `aenderung_cook`, um wie viel sich die Steigung ändert, wenn diese Schicht fehlt (neue minus alte Steigung).

In [ ]:
m_diag = smf.ols("ausbeute ~ stillstand_min * C(linie)", data=schichten).fit()
m_l3_diag = smf.ols("ausbeute ~ stillstand_min", data=t3).fit()

# TODO 10a
res_klassen = None
res_monat = None

# TODO 10b
vif_ausl = None

# TODO 10c
i_cook = None
cook_max = None
aenderung_cook = None

print(None if res_klassen is None else res_klassen.round(4))
zeige(vif_ausl, i_cook, cook_max, aenderung_cook)

# TODO 10a Grafik: links Residuen gegen Vorhersage, rechts Monatsmittel
fig_d, axes_d = plt.subplots(1, 2, figsize=(11, 4))

plt.show()

In [ ]:
check("TODO 10a Residuen je Klasse", lambda: res_klassen.shape == (5, 2) and float(res_klassen["mean"].abs().max()) < 0.003
      and np.allclose(res_klassen["std"].to_numpy(), [0.042967, 0.044152, 0.046538, 0.049342, 0.049793], atol=1e-5))
check("TODO 10a Residuen je Monat", lambda: len(res_monat) == 24 and math.isclose(float(res_monat.abs().max()), 0.0082015, abs_tol=1e-6))
check("TODO 10a Grafik", lambda: len(fig_d.axes) == 2 and len(axes_d[0].collections) >= 1 and len(axes_d[1].lines) >= 1)
check("TODO 10b VIF", lambda: math.isclose(vif_ausl, 1.3503178, abs_tol=1e-6))
check("TODO 10c einflussreichste Schicht", lambda: t3.iloc[i_cook]["datum"] == "2025-01-30" and t3.iloc[i_cook]["schicht"] == "Nacht"
      and math.isclose(cook_max, 0.0209368, abs_tol=1e-6) and math.isclose(aenderung_cook, 2.53558e-06, abs_tol=1e-9))

Die Residuen der Ausbeute zeigen keine Kurve, kein Monat weicht um mehr als 0,0082 ab, und die Streuung wächst nur leicht von 0,043 auf 0,050. Der VIF der Auslastung ist 1,35, ihr Standardfehler wächst durch das Personal um $\sqrt{1{,}35} = 1{,}16$. Die einflussreichste Schicht von Linie 3, die Nachtschicht am 30. Januar 2025, hatte bei 160 Minuten Stillstand nur 56 % Ausbeute. Ohne sie ändert sich die Steigung um 0,0000025 je Minute, das sind 0,25 %. Ob sie ein Erfassungsfehler ist, klärt erst ein Blick in die Störungsmeldungen dieser Schicht.

## 6 · Vorhersagen und ihre Grenzen
*Lerneinheit 6 · Skript Abschnitt 6*

`get_prediction(neu).summary_frame(alpha=0.05)` liefert das **Konfidenzintervall** für den Mittelwert (`mean_ci_lower`, `mean_ci_upper`) und das breitere **Vorhersageintervall** für eine einzelne Schicht (`obs_ci_lower`, `obs_ci_upper`). Geprüft wird außerhalb der Stichprobe: Schätzen mit dem ersten Jahr, Prüfen mit dem zweiten, mit $\text{MAE} = \frac1n\sum|y_i - \hat y_i|$ und $\text{RMSE} = \sqrt{\frac1n\sum(y_i - \hat y_i)^2}$, gegen ein einfaches Vergleichsmodell. Wie im Skript:

In [ ]:
neu = pd.DataFrame({"temp_z": [25 - 21], "linie": [1], "ausl_z": [0.90 - 0.85], "schicht": ["Früh"]})
print(m_z.get_prediction(neu).summary_frame(alpha=0.05).round(3))

jahr1 = schichten[schichten["datum"] < "2025-10-01"]
jahr2 = schichten[schichten["datum"] >= "2025-10-01"]
for formel in ["quote ~ C(linie)", "quote ~ temp_z * C(linie) + ausl_z + C(schicht)"]:
    m_j1 = smf.ols(formel, data=jahr1).fit()
    for daten in [jahr1, jahr2]:
        fehler = daten["quote"] - m_j1.predict(daten)
        print(len(daten), round(fehler.abs().mean(), 4), round(np.sqrt((fehler ** 2).mean()), 4))

### TODO 11 · Konfidenz- und Vorhersageintervall der Ausbeute
1. `vh_l3`: die Tabelle von `summary_frame(alpha=0.05)` für das Modell `ausbeute ~ stillstand_min` der Linie 3 bei 0, 120 und 480 Minuten Stillstand. `verh_breite`: wie viel breiter das Vorhersageintervall bei 120 Minuten ist als das Konfidenzintervall.
2. `abdeckung_pi` und `abdeckung_ci`: Schätze dasselbe Modell nur mit den Schichten von Linie 3 aus `jahr1`. Welcher Anteil der Schichten von Linie 3 aus `jahr2` liegt in seinem Vorhersageintervall, welcher in seinem Konfidenzintervall?

In [ ]:
# TODO 11a
vh_l3 = None
verh_breite = None

# TODO 11b
abdeckung_pi = None
abdeckung_ci = None

print(None if vh_l3 is None else vh_l3.round(4))
zeige(verh_breite, abdeckung_pi, abdeckung_ci)

In [ ]:
check("TODO 11a Intervalle", lambda: vh_l3.shape == (3, 6) and math.isclose(vh_l3.loc[1, "mean"], 0.8295701, abs_tol=1e-6)
      and math.isclose(vh_l3.loc[1, "obs_ci_lower"], 0.7397333, abs_tol=1e-6) and math.isclose(vh_l3.loc[2, "mean_ci_upper"], 0.4814843, abs_tol=1e-6))
check("TODO 11a Verhältnis der Breiten", lambda: math.isclose(verh_breite, 31.8031, abs_tol=1e-3))
check("TODO 11b Abdeckung im zweiten Jahr", lambda: math.isclose(abdeckung_pi, 697 / 730, abs_tol=1e-9) and math.isclose(abdeckung_ci, 54 / 730, abs_tol=1e-9))

Bei 120 Minuten Stillstand erreicht Linie 3 im Mittel 83,0 % Ausbeute, sicher auf 0,3 Prozentpunkte. Eine einzelne Schicht kann zwischen 74,0 % und 91,9 % landen, das Intervall ist 32-mal so breit. Im zweiten Jahr liegen 95,5 % der Schichten im Vorhersageintervall, wie versprochen, aber nur 7,4 % im Konfidenzintervall.

### TODO 12 · Zeitlich teilen und Überanpassung
1. `fehler_aus`: ein DataFrame mit dem Index `Mittelwert je Linie` und `Stillstand je Linie` und den Spalten `mae_jahr1`, `rmse_jahr1`, `mae_jahr2`, `rmse_jahr2`. Die Modelle `ausbeute ~ C(linie)` und `ausbeute ~ stillstand_min * C(linie)` werden mit `jahr1` geschätzt und auf beiden Jahren geprüft.
2. `rmse_kw`: ein DataFrame mit einer Zeile je Modell und den Spalten `koeffizienten`, `jahr1` und `jahr2` (RMSE). Die Modelle sind `ausbeute ~ stillstand_min * C(linie)` plus nacheinander nichts, `C(kw)`, `C(kw):C(linie)` und `C(kw):C(linie):C(schicht)`, wie im Skript.

In [ ]:
# TODO 12a
fehler_aus = None

# TODO 12b
rmse_kw = None

print(None if fehler_aus is None else fehler_aus.round(5))
print(None if rmse_kw is None else rmse_kw.round(5))

In [ ]:
check("TODO 12a Fehler beider Modelle", lambda: list(fehler_aus.index) == ["Mittelwert je Linie", "Stillstand je Linie"]
      and np.allclose(fehler_aus.loc["Mittelwert je Linie", ["mae_jahr1", "rmse_jahr1", "mae_jahr2", "rmse_jahr2"]].to_numpy(dtype=float), [0.063794, 0.083823, 0.063207, 0.082508], atol=2e-6)
      and np.allclose(fehler_aus.loc["Stillstand je Linie", ["mae_jahr1", "rmse_jahr1", "mae_jahr2", "rmse_jahr2"]].to_numpy(dtype=float), [0.037019, 0.047106, 0.036351, 0.046248], atol=2e-6))
check("TODO 12b Überanpassung", lambda: list(rmse_kw["koeffizienten"]) == [8, 57, 204, 604]
      and np.allclose(rmse_kw["jahr1"].to_numpy(dtype=float), [0.047106, 0.046703, 0.045620, 0.041803], atol=2e-6)
      and np.allclose(rmse_kw["jahr2"].to_numpy(dtype=float), [0.046248, 0.046608, 0.047567, 0.051013], atol=2e-6))

Der Stillstand senkt den Fehler gegenüber dem Mittelwert je Linie fast auf die Hälfte, im zweiten Jahr von 0,083 auf 0,046. Mit jeder Gruppe von Wochendummies sinkt der Fehler im ersten Jahr, im zweiten steigt er: 604 Koeffizienten passen sich den Zufällen des ersten Jahres an. Eine zufällige Teilung der Schichten mischt beide Jahre, dann sieht der Test anders aus, als das Modell später arbeiten muss.

## 7 · Regression ist keine Kausalität
*Lerneinheit 7 · Skript Abschnitt 7*

Ein Koeffizient beschreibt einen Zusammenhang unter den Bedingungen der Daten. Ein fehlender Confounder, eine falsche Kontrollvariable oder die falsche Richtung führen zu falschen Schlüssen über Ursachen. Die falsche Kontrollvariable und die umgekehrte Richtung wie im Skript:

In [ ]:
for formel in ["ausschuss_stueck ~ stillstand_min + C(linie)", "ausschuss_stueck ~ stillstand_min + C(linie) + ist_stueck"]:
    m_k = smf.ols(formel, data=schichten).fit()
    print(round(60 * m_k.params["stillstand_min"], 3), (60 * m_k.conf_int().loc["stillstand_min"]).round(3).tolist())

print(round(smf.ols("personal ~ auslastung + C(linie)", data=schichten).fit().params["auslastung"], 2),
      round(smf.ols("auslastung ~ personal + C(linie)", data=schichten).fit().params["personal"], 4))

### TODO 13 · Ein Modellbericht als Funktion
`modellbericht(formel, ziel, koeffizient)` fasst ein Modell zusammen und gibt ein Dictionary mit diesen Schlüsseln zurück:
- `n` und `r2_adj` aus dem Modell mit allen Schichten, `b`, `unten` und `oben`: der Koeffizient mit 95-%-Intervall aus robusten Standardfehlern (HC3),
- `rmse_modell`: der RMSE im zweiten Jahr, wenn das Modell mit dem ersten geschätzt wird, `rmse_vergleich`: dasselbe für `ziel ~ C(linie)`,
- `satz`: ein deutscher Satz mit Koeffizient, Intervall und beiden RMSE, Zahlen mit Dezimalkomma.

In [ ]:
# TODO 13
def modellbericht(formel, ziel, koeffizient):
    return None            # ersetze diese Zeile

bericht_aus = modellbericht("ausbeute ~ stillstand_min * C(linie)", "ausbeute", "stillstand_min")
bericht_temp = modellbericht("quote ~ temp_z * C(linie) + ausl_z + C(schicht)", "quote", "temp_z")
for b_ in [bericht_aus, bericht_temp]:
    print(None if b_ is None else b_["satz"])

In [ ]:
import re

SCHLUESSEL = {"n", "r2_adj", "b", "unten", "oben", "rmse_modell", "rmse_vergleich", "satz"}
check("TODO 13 Schlüssel", lambda: SCHLUESSEL <= set(bericht_aus) and SCHLUESSEL <= set(bericht_temp))
check("TODO 13 Ausbeute", lambda: bericht_aus["n"] == 5788 and math.isclose(bericht_aus["b"], -0.000791733, abs_tol=1e-9)
      and math.isclose(bericht_aus["unten"], -0.000812447, abs_tol=1e-9) and math.isclose(bericht_aus["oben"], -0.000771020, abs_tol=1e-9)
      and math.isclose(bericht_aus["rmse_modell"], 0.0462483, abs_tol=1e-6) and math.isclose(bericht_aus["rmse_vergleich"], 0.0825082, abs_tol=1e-6))
check("TODO 13 Temperatur", lambda: math.isclose(bericht_temp["r2_adj"], 0.3833222, abs_tol=1e-6) and math.isclose(bericht_temp["unten"], 0.1551225, abs_tol=1e-6)
      and math.isclose(bericht_temp["oben"], 0.2275607, abs_tol=1e-6) and math.isclose(bericht_temp["rmse_modell"], 1.0221155, abs_tol=1e-6))
check("TODO 13 Satz", lambda: all(isinstance(b_["satz"], str) and "HC3" in b_["satz"] and re.search(r"\d,\d", b_["satz"])
                                 and not re.search(r"\d\.\d{1,2}(?!\d)", b_["satz"]) for b_ in [bericht_aus, bericht_temp]))

Der Bericht nennt, was eine Leserin zum Nachprüfen braucht: den Koeffizienten mit robustem Intervall, die Zahl der Schichten, die Güte und die Prüfung im zweiten Jahr. Was er nicht sagen kann, ob eine Kühlung von Halle 1 den Ausschuss senkt, gehört als Grenze dazu.

## Bonus

### Bonus B1 · Bootstrap der Koeffizienten
Wie stark schwankt der Koeffizient der Auslastung im Modell `quote ~ auslastung + personal` für Linie 1, wenn du die Schichten neu ziehst?
1. Ziehe mit `rng` 1.000-mal 1.447 Schichten mit Zurücklegen (Zeilen von `X_se` und `y_se` aus TODO 8) und schätze jedes Mal mit `lstsq`.
2. `boot_se`: die Standardabweichung der 1.000 Koeffizienten der Auslastung. `boot_ki`: das 95-%-Perzentil-Intervall als Array `[unten, oben]`.

In [ ]:
# Bonus B1
rng = np.random.default_rng(101)
boot_se = None
boot_ki = None

zeige(boot_se, boot_ki)

In [ ]:
check("B1 Bootstrap", lambda: 0.445 <= boot_se <= 0.555 and len(boot_ki) == 2 and 0.6 <= boot_ki[0] <= 1.05 and 2.55 <= boot_ki[1] <= 3.05)

Der Bootstrap-Standardfehler liegt bei etwa 0,50, nah am robusten Wert 0,501 aus TODO 8 und etwas über dem klassischen 0,483. Das Perzentil-Intervall reicht von etwa 0,8 bis 2,8.

### Bonus B2 · Gewichtete Regression
Eine Quote aus vielen Teilen ist genauer als eine aus wenigen. Die **gewichtete Regression** `smf.wls(formel, data=..., weights=...)` gibt jeder Schicht ein Gewicht, hier die Zahl der gefertigten Stück.
1. `m_wls`: das Hauptmodell `quote ~ temp_z * C(linie) + ausl_z + C(schicht)` mit `weights=schichten["ist_stueck"]`. `steigung_wls`: eine Series mit der Temperatursteigung je Linie (Index 1 bis 4).
2. `streu_ols` und `streu_wls`: für Linie 1 die Standardabweichung der Residuen von `m_z` und die der gewichteten Residuen $\sqrt{n_i}\,e_i$ von `m_wls`, je in vier gleich großen Klassen von `ist_stueck` (`pd.qcut(..., 4, labels=False)`). `verh_ols` und `verh_wls`: jeweils die erste Klasse geteilt durch die letzte.

In [ ]:
# Bonus B2
m_wls = None
steigung_wls = None
streu_ols = None
streu_wls = None
verh_ols = None
verh_wls = None

print(None if steigung_wls is None else steigung_wls.round(4).to_dict())
zeige(verh_ols, verh_wls)

In [ ]:
check("B2 gewichtete Steigungen", lambda: np.allclose(steigung_wls.to_numpy(), [0.1979848, 0.2377488, 0.0520509, 0.0525899], atol=1e-6))
check("B2 Streuung je Klasse", lambda: len(streu_ols) == 4 and math.isclose(verh_ols, 1.21787, abs_tol=1e-4) and math.isclose(verh_wls, 1.01913, abs_tol=1e-4))

Gewichtet steigen die Quoten etwas stärker mit der Temperatur, in Linie 1 um 0,198 statt 0,191 Prozentpunkte je Grad. In Linie 1 streuen die ungewichteten Residuen kleiner Schichten 22 % stärker als die großer, die gewichteten fast gleich. Die Gewichte berücksichtigen aber nur die Stückzahl, nicht die Höhe der Quote, robuste Standardfehler bleiben deshalb sinnvoll.

## Zum Nachdenken

1. In TODO 1 und 6 kostet eine Stunde Stillstand Linie 3 mehr als die anderen Linien. Soll die Instandhaltung deshalb zuerst Linie 3 betreuen? Was müsstest du dafür noch wissen?
2. In TODO 4 ändert sich der Koeffizient der Auslastung mit dem Personal im Modell von 1,24 auf 1,81. Welche der beiden Zahlen würdest du der Werksleitung nennen, wenn sie fragt, was eine vollere Planung für den Ausschuss bedeutet?
3. In TODO 7 fehlen im Log-Modell 384 Schichten von Linie 4. Wie würdest du die Quote von Linie 4 stattdessen modellieren?
4. Die Ausbeute-Residuen in TODO 10 zeigen kaum Muster, und das Vorhersageintervall in TODO 11 hält, was es verspricht. Ist damit gezeigt, dass Stillstand die Ausbeute senkt?
5. Die Werksleitung erwägt eine Klimaanlage für Halle 1 und schickt dir den Bericht aus TODO 13 zurück: „Reicht das?“ Was antwortest du?

<details><summary>Hinweise zu den Antworten</summary>

1. Die Steigung sagt, wie viel Prozent der geplanten Stückzahl eine Stunde kostet. Für eine Entscheidung zählen auch, wie viele Stunden Stillstand jede Linie hat, was ein fehlendes Teil kostet und welche Störungen sich verhindern lassen. Linie 3 hat im Mittel den wenigsten Stillstand je Schicht.
2. Das Personal wird nach dem Plan eingeteilt. Fragt die Werksleitung nach der Wirkung einer volleren Planung samt dem Personal, das dazu eingeteilt wird, passt das kurze Modell ohne Personal, 1,24. Das lange Modell vergleicht Schichten mit gleichem Personal und beantwortet eine andere Frage.
3. Mit allen Schichten: etwa die Quote der Summen je Gruppe wie in Modul 2, ein lineares Modell der Quote mit robusten Standardfehlern wie im Skript oder ein Modell für Anteile wie die logistische Regression in Modul 8, die Nullen zulässt.
4. Nein. Gute Residuen zeigen, dass das Modell die Daten gut beschreibt. Ob Stillstand die Ausbeute senkt, ist hier zwar naheliegend, weil eine stehende Maschine nichts fertigt, aber das wissen wir aus dem Ablauf im Werk, nicht aus der Regression.
5. Der Bericht beschreibt einen Zusammenhang in beobachteten Daten zwischen 17,6 und 28,9 °C. Ob Kühlung wirkt, prüft am besten ein Versuch, etwa Kühlung an zufällig gewählten Tagen (Modul 12). Dazu kämen die Kosten der Anlage gegen den eingesparten Ausschuss.
</details>

## Wie es weitergeht

Du hast Geraden von Hand und mit statsmodels geschätzt, Koeffizienten bei sonst gleichen Größen gelesen, Kategorien und Wechselwirkungen eingebaut, Standardfehler klassisch und robust berechnet, Residuen gelesen und Vorhersagen außerhalb der Stichprobe geprüft. Bisher war die Zielgröße eine Zahl. **Modul 8 „Klassifikation und Modellbewertung“** fragt, was sich ändert, wenn die Zielgröße ja oder nein ist, etwa ob eine Schicht über 4 % Ausschuss kommt, und wie man Modelle mit Kreuzvalidierung fair vergleicht.